In [52]:
import pandas as pd
import re

In [53]:
df_mondo = pd.read_csv('../../../data/data/mondo/mondo_definitions_fix.csv')
df_mondo = df_mondo.rename({
    'name': 'name', 'definition': 'definition', 'id': 'mondo_id'
}, axis=1).astype({'mondo_id': int}).astype({'mondo_id': str})
df_mondo.head()

,mondo_id,name,definition
0,1,disease,A disease is a disposition to undergo patholog...
1,4,adrenocortical insufficiency,An endocrine or hormonal disorder that occurs ...
2,15,classic complement early component deficiency,A genetic deficiency of any early component of...
3,22,nocturnal enuresis,Urination during sleep.
4,44,hereditary hypophosphatemic rickets,Hypophosphatemic rickets is a group of genetic...


In [54]:
df_mondo.to_csv('../../../data/data_feature/disease_mondo.csv', index=False)

In [55]:
columns = ['CUI', 'unknown1', 'unknown2', 'space', 'source', 'description', 'unknown4','unknown5','unknown6']

df_umls = pd.read_csv('../../../data/data/umls/MRDEF.RRF', sep='|', low_memory=False, names=columns).get(['CUI', 'source', 'description'])
df_umls.head()

,CUI,source,description
0,C0000039,MSH,Synthetic phospholipid used in liposomes and l...
1,C0000039,MSHSWE,Syntetisk fosfolipid som används i liposomer o...
2,C0000039,MSHCZE,Syntetický fosfolipid používaný v liposomech a...
3,C0000039,MSHPOR,Fosfolipídeo sintético utilizado em lipossomos...
4,C0000039,MSHSPA,Fosfolípido sintético que se utiliza en liposo...


In [56]:
df_vocab = pd.read_csv('../../../data/data/vocab/umls_mondo.csv').astype({'mondo_id': int}).astype({'mondo_id': str})
df_vocab.head()

,umls_id,mondo_id
0,C0012634,1
1,C0405580,4
2,C0005818,9
3,C1285186,15
4,C0270327,22


In [59]:
df_umls_mondo = pd.merge(df_umls, df_vocab, how='inner', left_on='CUI', right_on='umls_id')
df_umls_mondo = df_umls_mondo.get(['mondo_id', 'umls_id', 'source', 'description'])

def fix_format(x):
    if pd.isna(x):
        return x
    s = []
    for word in x.split(' '):
        if len(word)>1 and word.isupper(): 
            s.append(word.lower())
        else: 
            s.append(word)
    s = " ".join(s)

    s = re.sub('\[.*?\]','',s)
    s = re.sub('\<.*?\>','',s)
    s = re.sub('\(.*?\)','',s)
    s = s.replace('  ', ' ')
    s = s.replace('  ', ' ')
    s = s.replace(' .', '')
    return s

df_umls_mondo.loc[:, 'description'] = [fix_format(x) for x in df_umls_mondo.get('description').values]
df_umls_mondo = df_umls_mondo.drop_duplicates().astype(str)
df_umls_mondo.head()

,mondo_id,umls_id,source,description
0,8692,C0000744,MSH,An autosomal recessive disorder of lipid metab...
1,8692,C0000744,CSP,disorder of lipid metabolism inherited as an a...
2,8692,C0000744,MSHSWE,Genetisk brist på betalipoprotein som leder ti...
3,8692,C0000744,NCI,An autosomal recessive disorder characterized ...
4,8692,C0000744,HPO,An absence of low-density lipoprotein choleste...


In [60]:
df_umls_mondo.to_csv('../../../data/data_feature/disease_umls.csv', index=False)